# AI-Powered Flood Risk Prediction
# Random Forest
## Ghale

Trains a Random Forest to classify flood risk at Murray Bridge. Uses `common.py` so the
data, features, label and chronological split are identical to the other models.

## 1. Load real data

In [1]:
import common
import joblib
from sklearn.ensemble import RandomForestClassifier

df = common.load_data()

print("Rows:", len(df))
print("Date range:", df["datetime"].min(), "to", df["datetime"].max())
df.head()

Rows: 6364
Date range: 2009-01-08 09:00:00 to 2026-07-02 09:00:00


,datetime,water_level_m,conductivity,water_temp_c
0,2009-01-08 09:00:00,-0.584,680.492,22.788
1,2009-01-09 09:00:00,-0.640,682.482,22.719
2,2009-01-10 09:00:00,-0.662,685.370,22.657
3,2009-01-11 09:00:00,-0.637,691.286,22.730
4,2009-01-12 09:00:00,-0.649,691.175,22.737


## 2. Feature engineering

In [2]:
X, y = common.build_features(df)

print("Risk threshold (m):", round(common.risk_threshold(df), 3))
print("Rows with features:", len(X))
y.value_counts()

Risk threshold (m): 0.806
Rows with features: 6357


high_risk
0    5083
1    1274
Name: count, dtype: int64

## 3. Train/test split

In [3]:
X_train, X_test, y_train, y_test = common.chronological_split(X, y)

print("Train:", len(X_train), "| Test:", len(X_test))

Train: 5403 | Test: 954


## 4. Train Random Forest

`max_depth=10` was chosen with an expanding-window backtest inside the training slice only
(train on the earlier part, score the next chunk, repeated at four cut points). The test
slice was not used to choose any setting. Depth is capped because fully grown trees overfit
the past and generalise worse to a later period.

In [4]:
model_rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
model_rf.fit(X_train, y_train)

pred_rf = model_rf.predict(X_test)
prob_rf = model_rf.predict_proba(X_test)[:, 1]

results = [
    common.evaluate("Random Forest", y_test, pred_rf, prob_rf),
    common.persistence_baseline(df, y_test),
]
common.comparison_table(results)

,F1,MCC,RMSE,Brier,NSE
model,,,,,
Random Forest,0.799,0.734,0.264,0.07,0.618
Persistence baseline,0.796,0.732,NaN,NaN,NaN


In [5]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, pred_rf, digits=3))
print("Confusion matrix (rows = actual):")
print(confusion_matrix(y_test, pred_rf))

              precision    recall  f1-score   support

           0      0.945     0.923     0.934       726
           1      0.771     0.829     0.799       228

    accuracy                          0.900       954
   macro avg      0.858     0.876     0.866       954
weighted avg      0.904     0.900     0.902       954

Confusion matrix (rows = actual):
[[670  56]
 [ 39 189]]


## 5. Feature importance

In [6]:
import pandas as pd

pd.Series(model_rf.feature_importances_, index=common.FEATURES).sort_values().round(3)

level_change3    0.055
level_lag2       0.158
level_roll7      0.289
level_lag1       0.499
dtype: float64

## 6. Save trained model

In [7]:
from pathlib import Path

repo = Path.cwd() if (Path.cwd() / "data").is_dir() else Path.cwd().parent
models_dir = repo / "models"
models_dir.mkdir(exist_ok=True)

joblib.dump(model_rf, models_dir / "random_forest.joblib")
print("Model saved to models/random_forest.joblib")

Model saved to models/random_forest.joblib
